In [2]:
from pyspark.sql import SparkSession

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \
    .appName("Pertemuan4-LatihanMandiri") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

26/09/16 22:19:57 WARN Utils: Your hostname, krock-PCPartner resolves to a loopback address: 127.0.1.1; using 10.194.253.236 instead (on interface wlp1s0)
26/09/16 22:19:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 22:20:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/16 22:20:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession berhasil dibuat!
Versi Spark: 3.5.9


In [9]:
import os
if not os.path.exists("data_transaksi_ecommerce.csv"):
    import numpy as np
    import pandas as pd
    np.random.seed(42)
    n = 600
    kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
    kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
    metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
    tanggal_range = pd.date_range("2026-07-01", "2026-07-31", freq="D")
    data = {
        "order_id": [f"ORD-{1000+i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25,0.25,0.20,0.15,0.15]),
        "kota": np.random.choice(kota_list, size=n),
        "unit_terjual": np.random.randint(1, 10, size=n),
        "harga_satuan": np.random.choice([25000,50000,75000,100000,150000,250000,500000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n, p=[0.35,0.30,0.20,0.15]),
    }
    pd.DataFrame(data).to_csv("data_transaksi_ecommerce.csv", index=False)
    print("Dataset dibuat ulang.")
else:
    print("Dataset sudah tersedia.")

Dataset dibuat ulang.


In [14]:
# Jalankan ini dulu sebelum mengerjakan latihan di bawah

df = spark.read.csv("data_transaksi_ecommerce.csv", header=True, inferSchema=True)
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))
print("Siap. Jumlah baris:", df.count())

print("Tipe Objek: ", type(df))
df.printSchema()

Siap. Jumlah baris: 600
Tipe Objek:  <class 'pyspark.sql.dataframe.DataFrame'>
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- total_pendapatan: integer (nullable = true)



In [15]:
from pyspark.sql.functions import col

df.filter(col("metode_pembayaran") == "E-Wallet") \
  .select("order_id", "kategori", "total_pendapatan") \
  .show(5)

+--------+--------------------+----------------+
|order_id|            kategori|total_pendapatan|
+--------+--------------------+----------------+
|ORD-1000|Kesehatan & Kecan...|         2250000|
|ORD-1003|          Elektronik|          150000|
|ORD-1006|   Makanan & Minuman|           25000|
|ORD-1013|   Makanan & Minuman|          900000|
|ORD-1014|             Fashion|           75000|
+--------+--------------------+----------------+
only showing top 5 rows



In [16]:
from pyspark.sql.functions import sum as spark_sum, col

ringkasan_kategori = df.groupBy("kategori").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan")
).orderBy(col("total_pendapatan").desc())

ringkasan_kategori.show()

[Stage 8:>                                                          (0 + 1) / 1]

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|             Fashion|       124825000|
|          Elektronik|       110600000|
|   Makanan & Minuman|        95400000|
|Kesehatan & Kecan...|        82200000|
|        Rumah Tangga|        77350000|
+--------------------+----------------+



In [17]:
df.groupBy("metode_pembayaran").count().show()

[Stage 11:>                                                         (0 + 1) / 1]

+-----------------+-----+
|metode_pembayaran|count|
+-----------------+-----+
|              COD|  114|
|    Transfer Bank|  208|
|     Kartu Kredit|   85|
|         E-Wallet|  193|
+-----------------+-----+

